In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow

In [2]:
BASE_DIR = Path().resolve().parents[0]

# DVF Data Cleaning

In [136]:
file_path = BASE_DIR / "data/raw/ValeursFoncieres-2025.txt"

In [137]:
cols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "No voie",
    "Type de voie",
    "Code voie",
    "Voie",
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Section",
    "No plan",
    "Code type local",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Surface terrain",
    "Nature culture",
    "Nature culture speciale"
]

dtype_map = {
    "Code postal": "str",
    "Code departement": "str",
    "Nature mutation": "category",
    "Type local": "category",
    "Type de voie": "category",
    "Commune": "category",
    "Nature culture": "str",
    "Nature culture speciale": "str",
}

In [138]:
df = pd.read_csv(file_path, sep="|", usecols=cols, dtype=dtype_map, parse_dates=["Date mutation"])

In [139]:
df.info(show_counts=True, memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 3714829 entries, 0 to 3714828
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype   
---  ------                     --------------    -----   
 0   Date mutation              3714829 non-null  str     
 1   Nature mutation            3714829 non-null  category
 2   Valeur fonciere            3666409 non-null  str     
 3   No voie                    2400906 non-null  float64 
 4   Type de voie               2323331 non-null  category
 5   Code voie                  3694165 non-null  str     
 6   Voie                       3694160 non-null  str     
 7   Code postal                3694049 non-null  str     
 8   Commune                    3714829 non-null  category
 9   Code departement           3714829 non-null  str     
 10  Code commune               3714829 non-null  int64   
 11  Section                    3714623 non-null  str     
 12  No plan                    3714829 non-null  int64   
 13  Code typ

## Normalisation, conversion and parsing

In [140]:
# Category
df["Nature mutation"] = (
    df["Nature mutation"].astype(str).str.strip().astype("category")
)

df["Commune"] = (
    df["Commune"].astype(str).str.strip().astype("category")
)

df["Type local"] = (
    df["Type local"].astype(str).str.strip().astype("category")
)

# Int
df["Valeur fonciere"] = (
    df["Valeur fonciere"].astype("str").str.replace(",", ".", regex=False)
)
df["Valeur fonciere"] = pd.to_numeric(df["Valeur fonciere"], errors="coerce")

# Str
df["Type de voie"] = (
    df["Type de voie"].astype(str).str.strip().astype("str")
)

df["Code voie"] = (
    df["Code voie"].astype(str).str.strip().astype("str")
)

df["Voie"] = (
    df["Voie"].astype(str).str.strip().astype("str")
)

df["Code postal"] = (
    df["Code postal"].astype(str).str.strip().astype("str")
)

df["Code departement"] = (
    df["Code departement"].astype(str).str.strip().astype("str")
)

df["Section"] = (
    df["Section"].astype(str).str.strip().astype("str")
)

df["Nature culture"] = (
    df["Nature culture"].astype(str).str.strip().astype("str")
)

df["Nature culture speciale"] = (
    df["Nature culture speciale"].astype(str).str.strip().astype("str")
)

# Date
df["Date mutation"] = pd.to_datetime(df["Date mutation"], format="%d/%m/%Y")


## NA values

In [141]:
df = df.dropna(subset=["Valeur fonciere"])

In [142]:
df = df.dropna(subset=["Surface reelle bati"])

In [143]:
df = df[df["Surface reelle bati"] != 0]

In [144]:
df = df.dropna(subset=["Code postal"])

In [145]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 1230775 entries, 2 to 3714827
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Date mutation              1230775 non-null  datetime64[us]
 1   Nature mutation            1230775 non-null  category      
 2   Valeur fonciere            1230775 non-null  float64       
 3   No voie                    1223753 non-null  float64       
 4   Type de voie               1115868 non-null  str           
 5   Code voie                  1230775 non-null  str           
 6   Voie                       1230774 non-null  str           
 7   Code postal                1230775 non-null  str           
 8   Commune                    1230775 non-null  category      
 9   Code departement           1230775 non-null  str           
 10  Code commune               1230775 non-null  int64         
 11  Section                    1230723 non-null  str     

## Use of category type

In [146]:
cat_cols_used = []
cat_cols = df.select_dtypes(include="category").columns

for col in cat_cols:
    nb_used = df[col].nunique()
    nb_total = len(df[col].cat.categories)

    cat_cols_used.append({
        "column": col,
        "used": nb_used,
        "unused": nb_total - nb_used,
        "total": nb_total
    })

report_df = pd.DataFrame(cat_cols_used)
report_df.sort_values("unused", ascending=False)
print(report_df)

            column   used  unused  total
0  Nature mutation      6       0      6
1          Commune  29131    1645  30776
2       Type local      3       1      4


### Unused category in "`Total local`"

'Dépendance' value is unused (count = 0)

In [147]:
df["Type local"].value_counts()

Type local
Maison                                      617328
Appartement                                 505017
Local industriel. commercial ou assimilé    108430
Dépendance                                       0
Name: count, dtype: int64

### Clean category (unused)

In [148]:
df["Type local"] = df["Type local"].cat.remove_unused_categories()

In [149]:
df["Commune"] = df["Commune"].cat.remove_unused_categories()

## Filter on "`Nature culture`" and "`Nature culture speciale`"

In [150]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 1230775 entries, 2 to 3714827
Data columns (total 20 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Date mutation              1230775 non-null  datetime64[us]
 1   Nature mutation            1230775 non-null  category      
 2   Valeur fonciere            1230775 non-null  float64       
 3   No voie                    1223753 non-null  float64       
 4   Type de voie               1115868 non-null  str           
 5   Code voie                  1230775 non-null  str           
 6   Voie                       1230774 non-null  str           
 7   Code postal                1230775 non-null  str           
 8   Commune                    1230775 non-null  category      
 9   Code departement           1230775 non-null  str           
 10  Code commune               1230775 non-null  int64         
 11  Section                    1230723 non-null  str     

In [151]:
df = df[
    df["Nature culture"].isna() |
    (df["Nature culture"].astype(str).str.strip() == "")
]

In [152]:
df = df[
    df["Nature culture speciale"].isna() |
    (df["Nature culture speciale"].astype(str).str.strip() == "")
]

In [153]:
df.drop(labels=["Nature culture", "Nature culture speciale", "Surface terrain"], axis="columns", inplace=True)

In [154]:
print(df.info(verbose=True, show_counts=True, memory_usage="deep"))

<class 'pandas.DataFrame'>
Index: 474711 entries, 18 to 3714827
Data columns (total 17 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Date mutation              474711 non-null  datetime64[us]
 1   Nature mutation            474711 non-null  category      
 2   Valeur fonciere            474711 non-null  float64       
 3   No voie                    470718 non-null  float64       
 4   Type de voie               459403 non-null  str           
 5   Code voie                  474711 non-null  str           
 6   Voie                       474710 non-null  str           
 7   Code postal                474711 non-null  str           
 8   Commune                    474711 non-null  category      
 9   Code departement           474711 non-null  str           
 10  Code commune               474711 non-null  int64         
 11  Section                    474685 non-null  str           
 12  No

## Handling anomalies

In [155]:
df = df[
    df["Valeur fonciere"].notna() &
    (df["Valeur fonciere"] >= 1)
]

df = df[
    df["Surface reelle bati"].notna() &
    (df["Surface reelle bati"] >= 1)
]

## Add new data

In [156]:
df["prix_m2"] = df["Valeur fonciere"] / df["Surface reelle bati"]

In [157]:
df["prix_m2"] = df["prix_m2"].replace(
    [np.inf, -np.inf],
    np.nan
)

df = df.dropna(subset=["prix_m2"])

In [158]:
df["prix_m2"].describe()

count    4.747100e+05
mean     1.797351e+04
std      1.731262e+05
min      2.622057e-05
25%      2.185714e+03
50%      3.442857e+03
75%      5.571429e+03
max      3.391250e+07
Name: prix_m2, dtype: float64

In [159]:
df = df[
    (df["prix_m2"] > df["prix_m2"].quantile(0.01)) &
    (df["prix_m2"] < df["prix_m2"].quantile(0.99))
]

In [160]:
df["prix_m2"].describe()

count    465206.000000
mean       6840.525667
std       18489.588594
min         161.538462
25%        2210.526316
50%        3442.857143
75%        5500.000000
max      324871.965986
Name: prix_m2, dtype: float64

## Final check

In [161]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
Index: 465206 entries, 18 to 3714827
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Date mutation              465206 non-null  datetime64[us]
 1   Nature mutation            465206 non-null  category      
 2   Valeur fonciere            465206 non-null  float64       
 3   No voie                    461241 non-null  float64       
 4   Type de voie               450313 non-null  str           
 5   Code voie                  465206 non-null  str           
 6   Voie                       465205 non-null  str           
 7   Code postal                465206 non-null  str           
 8   Commune                    465206 non-null  category      
 9   Code departement           465206 non-null  str           
 10  Code commune               465206 non-null  int64         
 11  Section                    465180 non-null  str           
 12  No

In [162]:
df.isna().sum().sort_values(ascending=False).head(20)

Type de voie                 14893
No voie                       3965
Section                         26
Voie                             1
Nature mutation                  0
Date mutation                    0
Code voie                        0
Code postal                      0
Commune                          0
Valeur fonciere                  0
Code departement                 0
Code commune                     0
No plan                          0
Code type local                  0
Type local                       0
Surface reelle bati              0
Nombre pieces principales        0
prix_m2                          0
dtype: int64

In [163]:
print("Length :", len(df))
print("Prix/m² médian :", df["prix_m2"].median())
print("Communes :", df["Commune"].nunique())

Length : 465206
Prix/m² médian : 3442.8571428571427
Communes : 9490


# Appartenance Commune Data Cleaning

In [ ]:
file_path = BASE_DIR / "data/raw/table-appartenance-geo-communes-2026.xlsx"

In [ ]:
cols = [
    "CODGEO",
    "LIBGEO",
    "DEP",
    "REG",
    "EPCI",
    "ZE2020"
]

dtype_map = {
    "CODGEO": "str",
    "LIBGEO": "str",
    "DEP": "str",
    "REG": "str",
    "EPCI": "str",
    "ZE2020": "str"
}

In [ ]:
df = pd.read_excel(file_path, engine="calamine", header=5, usecols=cols, dtype=dtype_map)
df

In [52]:
df.shape

(34875, 6)

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34875 entries, 0 to 34874
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   CODGEO  34875 non-null  str  
 1   LIBGEO  34875 non-null  str  
 2   DEP     34875 non-null  str  
 3   REG     34875 non-null  str  
 4   EPCI    34875 non-null  str  
 5   ZE2020  34875 non-null  str  
dtypes: str(6)
memory usage: 2.7 MB


In [54]:
df.nunique()

CODGEO    34875
LIBGEO    32645
DEP         101
REG          18
EPCI       1254
ZE2020      306
dtype: int64

# Stats Commune Data Cleaning

In [75]:
file_path = BASE_DIR / "data/raw/Stats_Commune.csv"

In [76]:
cols = [
    "Code",
    "Libellé",
    "Médiane du niveau de vie 2023",
    "Logements 2022",
    "Nb d'emplois au lieu de travail (LT) 2022",
    "Évol. annuelle moy. de la population 2017 - 2023 (en %)",
    "Évol. annuelle moy. de la pop. due au solde apparent entrées/sorties 2016-2022",
    "Unités légales (en nombre) 2023",
    "Créations d'entreprises (en nombre) 2025",
    "Nombre d'établissements 2024",
    "Effectifs salariés 2024",
    "École maternelle, primaire, élémentaire (en nombre) 2024",
    "Collège (en nombre) 2024",
    "Lycée (en nombre) 2024",
    "Pharmacie (en nombre) 2024",
    "Médecin généraliste (en nombre) 2024"
]

na_values = [
    "N/A - résultat non disponible",
    "N/A - secret statistique",
    "N/A - division par 0"
]

In [77]:
df = pd.read_csv(file_path, sep=";", header=2, usecols=cols, dtype=str, na_values=na_values)

In [78]:
df

,Code,Libellé,Médiane du niveau de vie 2023,Logements 2022,Nb d'emplois au lieu de travail (LT) 2022,Évol. annuelle moy. de la population 2017 - 2023 (en %),Évol. annuelle moy. de la pop. due au solde apparent entrées/sorties 2016-2022,Unités légales (en nombre) 2023,Créations d'entreprises (en nombre) 2025,Nombre d'établissements 2024,Effectifs salariés 2024,"École maternelle, primaire, élémentaire (en nombre) 2024",Collège (en nombre) 2024,Lycée (en nombre) 2024,Pharmacie (en nombre) 2024,Médecin généraliste (en nombre) 2024
0,01001,L'Abergement-Clémenciat,28270,379,77,1.73,1.5,68,8,19,53,1,0,0,0,0
1,01002,L'Abergement-de-Varey,28140,175,27,1.43,1.8,18,2,4,12,0,0,0,0,0
2,01004,Ambérieu-en-Bugey,24210,7973,8291,2.14,1.1,1130,280,592,5952,7,2,2,4,25
3,01005,Ambérieux-en-Dombes,28120,921,250,2.03,1.5,169,35,49,195,1,0,0,1,1
4,01006,Ambléon,NaN,71,5,0.59,0.4,5,2,1,3,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34870,97613,M'Tsangamouji,NaN,NaN,NaN,NaN,NaN,67,54,40,446,9,1,0,1,0
34871,97614,Ouangani,NaN,NaN,NaN,NaN,NaN,89,91,95,1545,9,1,2,1,3
34872,97615,Pamandzi,NaN,NaN,NaN,NaN,NaN,189,100,116,1471,9,1,1,1,3
34873,97616,Sada,NaN,NaN,NaN,NaN,NaN,174,126,128,1542,12,1,1,2,2


In [79]:
df.shape

(34875, 16)

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34875 entries, 0 to 34874
Data columns (total 16 columns):
 #   Column                                                                          Non-Null Count  Dtype
---  ------                                                                          --------------  -----
 0   Code                                                                            34875 non-null  str  
 1   Libellé                                                                         34875 non-null  str  
 2   Médiane du niveau de vie 2023                                                   30793 non-null  str  
 3   Logements 2022                                                                  34858 non-null  str  
 4   Nb d'emplois au lieu de travail (LT) 2022                                       34854 non-null  str  
 5   Évol. annuelle moy. de la population 2017 - 2023 (en %)                         34852 non-null  str  
 6   Évol. annuelle moy. de la pop. due au sol

In [81]:
df.nunique()

Code                                                                              34875
Libellé                                                                           32638
Médiane du niveau de vie 2023                                                      2202
Logements 2022                                                                     4315
Nb d'emplois au lieu de travail (LT) 2022                                          3513
Évol. annuelle moy. de la population 2017 - 2023 (en %)                            1159
Évol. annuelle moy. de la pop. due au solde apparent entrées/sorties 2016-2022      201
Unités légales (en nombre) 2023                                                    1582
Créations d'entreprises (en nombre) 2025                                            687
Nombre d'établissements 2024                                                       1039
Effectifs salariés 2024                                                            3361
École maternelle, primaire, élém

In [82]:
int_col = [
    "Logements 2022",
    "Nb d'emplois au lieu de travail (LT) 2022",
    "Unités légales (en nombre) 2023",
    "Créations d'entreprises (en nombre) 2025",
    "Nombre d'établissements 2024",
    "Effectifs salariés 2024",
    "École maternelle, primaire, élémentaire (en nombre) 2024",
    "Collège (en nombre) 2024",
    "Lycée (en nombre) 2024",
    "Pharmacie (en nombre) 2024",
    "Médecin généraliste (en nombre) 2024"
]

float_col = [
    "Médiane du niveau de vie 2023",
    "Évol. annuelle moy. de la population 2017 - 2023 (en %)",
    "Évol. annuelle moy. de la pop. due au solde apparent entrées/sorties 2016-2022"
]

for col in int_col:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

for col in float_col:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34875 entries, 0 to 34874
Data columns (total 16 columns):
 #   Column                                                                          Non-Null Count  Dtype  
---  ------                                                                          --------------  -----  
 0   Code                                                                            34875 non-null  str    
 1   Libellé                                                                         34875 non-null  str    
 2   Médiane du niveau de vie 2023                                                   30793 non-null  float64
 3   Logements 2022                                                                  34858 non-null  Int64  
 4   Nb d'emplois au lieu de travail (LT) 2022                                       34854 non-null  Int64  
 5   Évol. annuelle moy. de la population 2017 - 2023 (en %)                         34852 non-null  float64
 6   Évol. annuelle moy. de la

In [83]:
cols_to_check = df.columns.difference(["Code", "Libellé"])

rows_all_nan = df[df[cols_to_check].isna().all(axis=1)]

rows_all_nan

,Code,Libellé,Médiane du niveau de vie 2023,Logements 2022,Nb d'emplois au lieu de travail (LT) 2022,Évol. annuelle moy. de la population 2017 - 2023 (en %),Évol. annuelle moy. de la pop. due au solde apparent entrées/sorties 2016-2022,Unités légales (en nombre) 2023,Créations d'entreprises (en nombre) 2025,Nombre d'établissements 2024,Effectifs salariés 2024,"École maternelle, primaire, élémentaire (en nombre) 2024",Collège (en nombre) 2024,Lycée (en nombre) 2024,Pharmacie (en nombre) 2024,Médecin généraliste (en nombre) 2024


# Stats Intercommunalite Data Cleaning

In [55]:
file_path = BASE_DIR / "data/raw/Stats_Interco.csv"

In [64]:
df = pd.read_csv(file_path, sep=";", header=2, dtype=str, na_values=na_values)

In [65]:
df

,Code,Libellé,Salaire net EQTP mensuel moyen 2023,Taux de pauvreté 2023
0,200000172,Communauté de communes Faucigny-Glières,2633,9.6
1,200000438,Communauté de communes du Pays de Pontchâteau ...,2202,8.3
2,200000545,Communauté de communes des Portes de Romilly-s...,2169,27
3,200000628,Communauté de communes Rhône Lez Provence,2193,22.6
4,200000800,Communauté de communes Cœur de Sologne,2171,11
...,...,...,...,...
1250,249740077,Communauté d'agglomération CIVIS (Communauté I...,2138,36.4
1251,249740085,Communauté d'agglomération du Sud,1955,39.6
1252,249740093,Communauté d'agglomération Intercommunale de l...,1972,43.9
1253,249740101,Communauté d'agglomération Territoire de la Cô...,2344,32.2


In [66]:
df.shape

(1255, 4)

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1255 entries, 0 to 1254
Data columns (total 4 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   Code                                 1255 non-null   str  
 1   Libellé                              1255 non-null   str  
 2   Salaire net EQTP mensuel moyen 2023  1250 non-null   str  
 3   Taux de pauvreté 2023                1223 non-null   str  
dtypes: str(4)
memory usage: 115.3 KB


In [68]:
df.nunique()

Code                                   1255
Libellé                                1253
Salaire net EQTP mensuel moyen 2023     601
Taux de pauvreté 2023                   200
dtype: int64

In [69]:
df["Salaire net EQTP mensuel moyen 2023"] = pd.to_numeric(df["Salaire net EQTP mensuel moyen 2023"], errors="coerce")
df["Taux de pauvreté 2023"] = pd.to_numeric(df["Taux de pauvreté 2023"], errors="coerce")

In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1255 entries, 0 to 1254
Data columns (total 4 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Code                                 1255 non-null   str    
 1   Libellé                              1255 non-null   str    
 2   Salaire net EQTP mensuel moyen 2023  1250 non-null   float64
 3   Taux de pauvreté 2023                1223 non-null   float64
dtypes: float64(2), str(2)
memory usage: 106.8 KB


In [71]:
df

,Code,Libellé,Salaire net EQTP mensuel moyen 2023,Taux de pauvreté 2023
0,200000172,Communauté de communes Faucigny-Glières,2633.0,9.6
1,200000438,Communauté de communes du Pays de Pontchâteau ...,2202.0,8.3
2,200000545,Communauté de communes des Portes de Romilly-s...,2169.0,27.0
3,200000628,Communauté de communes Rhône Lez Provence,2193.0,22.6
4,200000800,Communauté de communes Cœur de Sologne,2171.0,11.0
...,...,...,...,...
1250,249740077,Communauté d'agglomération CIVIS (Communauté I...,2138.0,36.4
1251,249740085,Communauté d'agglomération du Sud,1955.0,39.6
1252,249740093,Communauté d'agglomération Intercommunale de l...,1972.0,43.9
1253,249740101,Communauté d'agglomération Territoire de la Cô...,2344.0,32.2


In [72]:
df[
    (df["Salaire net EQTP mensuel moyen 2023"].isna()) &
    (df["Taux de pauvreté 2023"].isna())
]

,Code,Libellé,Salaire net EQTP mensuel moyen 2023,Taux de pauvreté 2023
268,200050532,Communauté de communes de Petite-Terre,NaN,NaN
286,200059871,Communauté de communes du Centre-Ouest,NaN,NaN
289,200060457,Communauté d'agglomération de Dembeni / Mamoudzou,NaN,NaN
290,200060465,Communauté d'agglomération du Grand Nord de Ma...,NaN,NaN
291,200060473,Communauté de communes du Sud,NaN,NaN


# Stats Chomage Data Cleaning

In [84]:
file_path = BASE_DIR / "data/raw/Stats_Chomage.csv"

In [85]:
df = pd.read_csv(file_path, sep=";", header=2, dtype=str, na_values=na_values)

In [86]:
df

,Code,Libellé,Taux de chômage trimestriel 2025-T4
0,0051,Alençon,7.2
1,0052,Arles,9.8
2,0053,Avignon,10.8
3,0054,Beauvais,8.2
4,0055,Bollène-Pierrelatte,9.4
...,...,...,...
301,9403,Calvi,7.7
302,9404,Corte,5.8
303,9405,Ghisonaccia,8.1
304,9406,Porto-Vecchio,8.6


In [87]:
df.shape

(306, 3)

In [88]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 3 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   Code                                 306 non-null    str  
 1   Libellé                              306 non-null    str  
 2   Taux de chômage trimestriel 2025-T4  302 non-null    str  
dtypes: str(3)
memory usage: 12.5 KB


In [91]:
df["Taux de chômage trimestriel 2025-T4"] = pd.to_numeric(df["Taux de chômage trimestriel 2025-T4"], errors="coerce")

In [92]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 3 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Code                                 306 non-null    str    
 1   Libellé                              306 non-null    str    
 2   Taux de chômage trimestriel 2025-T4  302 non-null    float64
dtypes: float64(1), str(2)
memory usage: 11.6 KB


In [90]:
df[df["Taux de chômage trimestriel 2025-T4"].isna()]

,Code,Libellé,Taux de chômage trimestriel 2025-T4
25,0301,Est-littoral,NaN
26,0302,Ouest-Guyanais,NaN
27,0303,Savanes,NaN
32,0601,Mayotte,NaN
